# Huấn luyện YOLOv8 cho nhận diện biển báo (Zalo AI 2020)
Notebook này thực hiện từ A-Z: Chuyển đổi dữ liệu JSON sang định dạng YOLO, chia tập Train/Val, và tiến hành huấn luyện.

In [ ]:
import os
import json
import glob
import shutil
import random
from tqdm import tqdm

# Thiết lập các đường dẫn thư mục
# Tìm file json tự động trong /kaggle/input/ để tránh lỗi sai đường dẫn
json_paths = glob.glob('/kaggle/input/**/train_traffic_sign_dataset.json', recursive=True)
if not json_paths:
    raise FileNotFoundError("Không tìm thấy file JSON. Vui lòng kiểm tra lại dataset đã add vào Kaggle chưa!")

json_path = json_paths[0]
image_dir = os.path.dirname(json_path).replace('traffic_train', 'traffic_train/images')

# Nếu cách nối chuỗi trên không tìm thấy thư mục ảnh, dùng lệnh tìm kiếm thư mục cho an toàn
if not os.path.exists(image_dir):
    img_dirs = glob.glob('/kaggle/input/**/traffic_train/images', recursive=True)
    if img_dirs:
        image_dir = img_dirs[0]

dataset_dir = '/kaggle/working/dataset'

# Tạo cấu trúc thư mục YOLO chuẩn (chia 2 tập train và val)
for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/{split}/images', exist_ok=True)
    os.makedirs(f'{dataset_dir}/{split}/labels', exist_ok=True)

# Bắt đầu đọc file JSON và nạp dữ liệu
print("Đang đọc dữ liệu JSON...")
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_info = {img['id']: img for img in data['images']}

# Gom các bounding box lại theo từng bức ảnh để tiện cho việc xử lý về sau
img_to_anns = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in img_to_anns:
        img_to_anns[img_id] = []
    img_to_anns[img_id].append(ann)

# Phân chia tập dữ liệu ngẫu nhiên thành Train (80%) và Val (20%)
image_ids = list(images_info.keys())
random.seed(42)
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.8)
train_ids = image_ids[:split_idx]
val_ids = image_ids[split_idx:]

print(f"Tổng số ảnh: {len(image_ids)}. Train: {len(train_ids)}, Val: {len(val_ids)}")

In [ ]:
# Hàm chuyển đổi tọa độ từ COCO sang chuẩn YOLO
def convert_coco_to_yolo(bbox, img_width, img_height):
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return x_center, y_center, w_norm, h_norm

# Thực hiện vòng lặp cắt chuyển đổi và copy ảnh vào đúng thư mục
print("Đang xử lý và tạo file .txt cho YOLO...")

def process_split(ids, split_name):
    for img_id in tqdm(ids, desc=f"Processing {split_name}"):
        img_info = images_info[img_id]
        img_filename = img_info['file_name']
        img_width = img_info['width']
        img_height = img_info['height']
        
        src_img_path = os.path.join(image_dir, img_filename)
        dst_img_path = os.path.join(dataset_dir, split_name, 'images', img_filename)
        
        # Nếu ảnh thực sự tồn tại thì copy sang thư mục YOLO và tạo file nhãn
        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
            
            txt_filename = img_filename.rsplit('.', 1)[0] + '.txt'
            txt_path = os.path.join(dataset_dir, split_name, 'labels', txt_filename)
            
            with open(txt_path, 'w') as f_txt:
                if img_id in img_to_anns:
                    for ann in img_to_anns[img_id]:
                        # YOLO class index bắt đầu từ 0, nên id gốc (1->7) phải trừ đi 1
                        class_id = int(ann['category_id']) - 1
                        x_c, y_c, w_n, h_n = convert_coco_to_yolo(ann['bbox'], img_width, img_height)
                        f_txt.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

process_split(train_ids, 'train')
process_split(val_ids, 'val')

print("Hoàn tất quá trình chuẩn bị dữ liệu!")

In [ ]:
# Tạo file cấu hình dataset.yaml cho mô hình YOLO đọc
yaml_content = f"""
path: {dataset_dir}
train: train/images
val: val/images

# Danh sách 7 loại biển báo (class_id tương ứng từ 0 tới 6)
names:
  0: No entry
  1: No parking / waiting
  2: No turning
  3: Max Speed
  4: Other prohibition signs
  5: Warning signs
  6: Mandatory signs
"""

with open('/kaggle/working/dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())
    
print("Đã tạo xong file cấu hình dataset.yaml")

In [ ]:
# [TỪ EDA E3] TỰ ĐỘNG TẠO KIẾN TRÚC YOLOv8-P2 CHO VẬT THỂ SIÊU NHỎ
import yaml
from ultralytics import YOLO

p2_yaml_content = """
# Ultralytics YOLO 🚀, AGPL-3.0 license
# YOLOv8-p2 architecture
nc: 7  # number of classes
scales: 
  s: [0.33, 0.50, 1024] 

backbone:
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]  # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]  # cat backbone P4
  - [-1, 3, C2f, [512]]  # 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]  # cat backbone P3
  - [-1, 3, C2f, [256]]  # 15 (P3/8-small)

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]  # cat backbone P2
  - [-1, 3, C2f, [128]]  # 18 (P2/4-xsmall)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]  # cat head P3
  - [-1, 3, C2f, [256]]  # 21 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]  # cat head P4
  - [-1, 3, C2f, [512]]  # 24 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]  # cat head P5
  - [-1, 3, C2f, [1024]]  # 27 (P5/32-large)

  - [[18, 21, 24, 27], 1, Detect, [nc]]  # Detect(P2, P3, P4, P5)
"""

with open('/kaggle/working/yolov8s-p2.yaml', 'w', encoding='utf-8') as f:
    f.write(p2_yaml_content)
print("Đã tạo kiến trúc mạng YOLOv8s-P2 thành công!")


In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình kiến trúc P2
model = YOLO('/kaggle/working/yolov8s-p2.yaml')
model.load('yolov8s.pt') # Load pretrained để học nhanh

# Thiết lập Hyperparameters khổng lồ đúc kết từ EDA
results = model.train(
    data='/kaggle/working/dataset.yaml',
    epochs=50,
    imgsz=1280,         # [E3] High-res bảo toàn pixel vật thể nhỏ
    batch=8,
    max_det=50,         # [E1, E2] Tối ưu luồng NMS, tăng tốc luồng xử lý
    iou=0.6,            # [E2] Giữ biển báo đứng cạnh nhau
    optimizer='AdamW',  # [M4.4] Lịch trình học thuật
    cos_lr=True,        # [M4.4] Hạ nhiệt độ Learning Rate bằng Cosine Annealing
    
    # --- Kỹ thuật Hàm Loss ---
    fl_gamma=2.0,       # [E1] Kích hoạt Focal Loss trị Imbalanced Data
    cls=2.0,            # [M4.1] Tăng cls_gain, ép soi kỹ hình vẽ bên trong biển báo
    box=1.0,
    
    # --- Augmentation & Khắc phục Center Bias ---
    mosaic=1.0,         # Trộn 4 ảnh
    degrees=10.0,       # Xoay nhẹ
    translate=0.2,      # [E4] Random Shift văng biển báo ra mép ảnh phá Center Bias
    
    project='/kaggle/working/zalo_traffic',
    name='yolov8s_p2_highres',
    device=0,  # Kích hoạt GPU
)



In [ ]:
# TỰ ĐỘNG NÉN TOÀN BỘ KẾT QUẢ ĐỂ TẢI VỀ (Chỉ chạy sau khi ô phía trên đã huấn luyện xong)
# Nén toàn bộ thư mục zalo_traffic thành 1 cục duy nhất tên là yolo_results.zip
print("Đang nén kết quả để tải về. Vui lòng đợi...")
!zip -r -q /kaggle/working/yolo_results.zip /kaggle/working/zalo_traffic
print("Đã nén xong! Hãy nhìn sang cột Output bên phải, bạn sẽ thấy file yolo_results.zip để tải về máy tính.")

In [ ]:
# [TỪ E3, M4.2] THUẬT TOÁN DÀNH CHO WEB APP
!pip install -q sahi
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

try:
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path='/kaggle/working/zalo_traffic/yolov8s_p2_highres/weights/best.pt',
        confidence_threshold=0.25,
        device="cuda:0"
    )

    # Dùng SAHI cắt nhỏ ảnh và dự đoán kết hợp Soft-NMS
    # Note: Đây là mã giả lập, cần một ảnh test có sẵn để chạy lệnh này
    print("Hệ thống SAHI đã sẵn sàng cho Web App. Sử dụng get_sliced_prediction() để dự đoán ảnh thực tế.")
except Exception as e:
    print("Chưa có weights để chạy SAHI:", e)
